In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import json
import time
import os
import pickle

def setup_driver():
    """Set up a Chrome driver with enhanced bot detection evasion."""
    chrome_options = Options()
    # Headless mode disabled for debugging
    # chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.execute_cdp_cmd("Network.setUserAgentOverride", {
        "userAgent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    })
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def normalize_champion_name(champ):
    """Normalize champion names to match LoLalytics URL format (lowercase, no spaces or special characters)."""
    champ = champ.lower().replace(" ", "").replace("'", "").replace(".", "")
    special_cases = {
        "nunu&willump": "nunu",
        "drmundo": "drmundo",
        "ksante": "ksante",
        "aurelionsol": "aurelionsol",
        "jarvaniv": "jarvaniv",
        "leblanc": "leblanc",
        "leesin": "leesin",
        "twistedfate": "twistedfate",
        "velkoz": "velkoz",
        "tahmkench": "tahmkench",
        "belveth": "belveth",
        "renataglasc": "renataglasc"
    }
    return special_cases.get(champ, champ)

def scrape_champion_data(driver, champion, max_retries=3):
    """Scrape ADC matchup (Weak Against) and good synergy data from Common Matchups and Synergies tab."""
    normalized_champ = normalize_champion_name(champion)
    url = f"https://lolalytics.com/lol/{normalized_champ}/build/"
    for attempt in range(max_retries):
        try:
            driver.get(url)
            # Wait for the page to load
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div[id='root'], body"))
            )
            # Save initial page source
            with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_initial.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"Saved initial page source for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_initial.html")
            # Save screenshot
            driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_initial.png")
            print(f"Saved initial screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_initial.png")

            data = {"champion": champion, "matchups": [], "synergies": []}

            # Click the "Common Matchups and Synergies" tab
            try:
                tab_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.XPATH, "//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'matchups') or contains(@class, 'tab') or contains(@class, 'matchup')]"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", tab_button)
                driver.execute_script("arguments[0].click();", tab_button)
                print(f"Clicked 'Common Matchups and Synergies' tab for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_tab.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after tab click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_tab.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_tab.png")
                print(f"Saved tab screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_tab.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Common Matchups and Synergies' tab for {champion}: {e}")

            # Click the "Weak Against" button for matchups
            try:
                weak_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-type='weak_counter']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", weak_button)
                driver.execute_script("arguments[0].click();", weak_button)
                print(f"Clicked 'Weak Against' button for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_weak.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after weak button click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_weak.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_weak.png")
                print(f"Saved weak button screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_weak.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Weak Against' button for {champion}: {e}")

            # Scrape matchups (Weak Against filter)
            try:
                matchup_table = WebDriverWait(driver, 30).until(
                    EC.visibility_of_element_located((By.CSS_SELECTOR, "table[class*='matchup'], div[class*='matchup'] table"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", matchup_table)
                rows = matchup_table.find_elements(By.TAG_NAME, "tr")
                print(f"Found {len(rows)} rows in matchup table for {champion}")
                for row in rows[1:]:  # Skip header row
                    cols = row.find_elements(By_TAG_NAME, "td")
                    if len(cols) >= 3:
                        matchup_data = {
                            "opponent": cols[0].text.strip(),
                            "win_rate": cols[1].text.strip(),
                            "games": cols[2].text.strip()
                        }
                        data["matchups"].append(matchup_data)
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error scraping matchups for {champion}: {e}")
                # Log table HTML if found
                try:
                    tables = driver.find_elements(By.CSS_SELECTOR, "table[class*='matchup'], div[class*='matchup'] table")
                    for i, table in enumerate(tables):
                        print(f"Matchup table {i} HTML: {table.get_attribute('outerHTML')[:200]}...")
                except:
                    print("No matchup tables found for logging.")

            # Click the "Good Synergy" button for synergies
            try:
                synergy_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-type='good_synergy']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", synergy_button)
                driver.execute_script("arguments[0].click();", synergy_button)
                print(f"Clicked 'Good Synergy' button for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_synergy.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after synergy click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_synergy.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_synergy.png")
                print(f"Saved synergy screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_synergy.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Good Synergy' button for {champion}: {e}")

            # Scrape synergies (Good Synergy filter)
            try:
                synergy_table = WebDriverWait(driver, 30).until(
                    EC.visibility_of_element_located((By.CSS_SELECTOR, "table[class*='synergy'], div[class*='synergy'] table"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", synergy_table)
                rows = synergy_table.find_elements(By_TAG_NAME, "tr")
                print(f"Found {len(rows)} rows in synergy table for {champion}")
                for row in rows[1:]:  # Skip header row
                    cols = row.find_elements(By_TAG_NAME, "td")
                    if len(cols) >= 3:
                        synergy_data = {
                            "ally": cols[0].text.strip(),
                            "win_rate": cols[1].text.strip(),
                            "games": cols[2].text.strip()
                        }
                        data["synergies"].append(synergy_data)
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error scraping synergies for {champion}: {e}")
                # Log table HTML if found
                try:
                    tables = driver.find_elements(By.CSS_SELECTOR, "table[class*='synergy'], div[class*='synergy'] table")
                    for i, table in enumerate(tables):
                        print(f"Synergy table {i} HTML: {table.get_attribute('outerHTML')[:200]}...")
                except:
                    print("No synergy tables found for logging.")

            if data["matchups"] or data["synergies"]:
                return data
            print(f"Attempt {attempt + 1}: No data scraped for {champion}, retrying...")
            time.sleep(5)

        except Exception as e:
            print(f"Attempt {attempt + 1}: Error loading page for {champion}: {e}")
            with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_error.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"Saved page source for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_error.html")
            driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_error.png")
            print(f"Saved error screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_error.png")
            time.sleep(5)

    print(f"Failed to scrape data for {champion} after {max_retries} attempts.")
    return {"champion": champion, "matchups": [], "synergies": []}

def main():
    driver = setup_driver()
    try:
        # Load champions from pickle file
        try:
            with open("champions.pkl", "rb") as champion_file:
                champions = pickle.load(champion_file)
        except FileNotFoundError:
            print("Error: champions.pkl not found. Exiting.")
            return
        except pickle.PickleError as e:
            print(f"Error: Failed to load champions.pkl: {e}. Exiting.")
            return

        # Validate champion list
        champions = list(dict.fromkeys(champions))  # Remove duplicates
        if not champions:
            print("No champions found. Exiting.")
            return
        print(f"Found {len(champions)} champions: {champions}")

        # Check for unexpected champion count
        expected_count = 171  # Updated to expect 171 champions
        if len(champions) != expected_count:
            print(f"Warning: Expected {expected_count} champions, but found {len(champions)}. Please verify champions.pkl.")

        all_data = []
        total_champions = len(champions)
        for i, champion in enumerate(champions, 1):
            print(f"Scraping data for {champion} ({i}/{total_champions})...")
            champ_data = scrape_champion_data(driver, champion)
            all_data.append(champ_data)
            # Save partial data every 10 champions
            if i % 10 == 0:
                with open("lolalytics_champion_data_partial.json", "w", encoding="utf-8") as f:
                    json.dump(all_data, f, indent=2)
                print(f"Partial data saved to lolalytics_champion_data_partial.json after {i} champions")
            time.sleep(3)

        # Save final data
        with open("lolalytics_champion_data.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, indent=2)
        print("Data saved to lolalytics_champion_data.json")

    finally:
        driver.quit()

if __name__ == "__main__":
    main()

Found 171 champions: ['aatrox', 'ahri', 'akali', 'akshan', 'alistar', 'ambessa', 'amumu', 'anivia', 'annie', 'aphelios', 'ashe', 'aurelionsol', 'aurora', 'azir', 'bard', 'belveth', 'blitzcrank', 'brand', 'braum', 'briar', 'caitlyn', 'camille', 'cassiopeia', 'chogath', 'corki', 'darius', 'diana', 'drmundo', 'draven', 'ekko', 'elise', 'evelynn', 'ezreal', 'fiddlesticks', 'fiora', 'fizz', 'galio', 'gangplank', 'garen', 'gnar', 'gragas', 'graves', 'gwen', 'hecarim', 'heimerdinger', 'hwei', 'illaoi', 'irelia', 'ivern', 'janna', 'jarvaniv', 'jax', 'jayce', 'jhin', 'jinx', 'ksante', 'kaisa', 'kalista', 'karma', 'karthus', 'kassadin', 'katarina', 'kayle', 'kayn', 'kennen', 'khazix', 'kindred', 'kled', 'kogmaw', 'leblanc', 'leesin', 'leona', 'lillia', 'lissandra', 'lucian', 'lulu', 'lux', 'malphite', 'malzahar', 'maokai', 'masteryi', 'mel', 'milio', 'missfortune', 'mordekaiser', 'morgana', 'naafiri', 'nami', 'nasus', 'nautilus', 'neeko', 'nidalee', 'nilah', 'nocturne', 'nunu&willump', 'olaf', '

KeyboardInterrupt: 

In [3]:
import pickle
champions = [
    "Aatrox", "Ahri", "Akali", "Akshan", "Alistar", "Ambessa", "Amumu", "Anivia", "Annie", "Aphelios",
    "Ashe", "Aurelion Sol", "Aurora", "Azir", "Bard", "Belveth", "Blitzcrank", "Brand", "Braum", "Briar",
    "Caitlyn", "Camille", "Cassiopeia", "Chogath", "Corki", "Darius", "Diana", "Dr. Mundo", "Draven", "Ekko",
    "Elise", "Evelynn", "Ezreal", "Fiddlesticks", "Fiora", "Fizz", "Galio", "Gangplank", "Garen", "Gnar",
    "Gragas", "Graves", "Gwen", "Hecarim", "Heimerdinger", "Hwei", "Illaoi", "Irelia", "Ivern", "Janna",
    "Jarvan IV", "Jax", "Jayce", "Jhin", "Jinx", "K'Sante", "Kai'Sa", "Kalista", "Karma", "Karthus",
    "Kassadin", "Katarina", "Kayle", "Kayn", "Kennen", "Kha'Zix", "Kindred", "Kled", "Kog'Maw", "LeBlanc",
    "Lee Sin", "Leona", "Lillia", "Lissandra", "Lucian", "Lulu", "Lux", "Malphite", "Malzahar", "Maokai",
    "Master Yi", "Mel", "Milio", "Miss Fortune", "Mordekaiser", "Morgana", "Naafiri", "Nami", "Nasus",
    "Nautilus", "Neeko", "Nidalee", "Nilah", "Nocturne", "Nunu & Willump", "Olaf", "Orianna", "Ornn",
    "Pantheon", "Poppy", "Pyke", "Qiyana", "Quinn", "Rakan", "Rammus", "Rek'Sai", "Rell", "Renata Glasc",
    "Renekton", "Rengar", "Riven", "Rumble", "Ryze", "Samira", "Sejuani", "Senna", "Seraphine", "Sett",
    "Shaco", "Shen", "Shyvana", "Singed", "Sion", "Sivir", "Skarner", "Smolder", "Sona", "Soraka",
    "Swain", "Sylas", "Syndra", "Tahm Kench", "Taliyah", "Talon", "Taric", "Teemo", "Thresh", "Tristana",
    "Trundle", "Tryndamere", "Twisted Fate", "Twitch", "Udyr", "Urgot", "Varus", "Vayne", "Veigar",
    "Vel'Koz", "Vex", "Vi", "Viego", "Viktor", "Vladimir", "Volibear", "Warwick", "Wukong", "Xayah",
    "Xerath", "Xin Zhao", "Yasuo", "Yone", "Yorick", "Yunara", "Yuumi", "Zac", "Zed", "Zeri", "Ziggs",
    "Zilean", "Zoe", "Zyra"
]
champions = [champion.lower() for champion in champions]
champions = [champion.replace(" ", "") for champion in champions]
champions = [champion.replace("'", "") for champion in champions]
champions = [champion.replace(".", "") for champion in champions]
print(champions)
with open("champions.pkl", "wb") as champion_file:
    pickle.dump(champions, champion_file)
    

['aatrox', 'ahri', 'akali', 'akshan', 'alistar', 'ambessa', 'amumu', 'anivia', 'annie', 'aphelios', 'ashe', 'aurelionsol', 'aurora', 'azir', 'bard', 'belveth', 'blitzcrank', 'brand', 'braum', 'briar', 'caitlyn', 'camille', 'cassiopeia', 'chogath', 'corki', 'darius', 'diana', 'drmundo', 'draven', 'ekko', 'elise', 'evelynn', 'ezreal', 'fiddlesticks', 'fiora', 'fizz', 'galio', 'gangplank', 'garen', 'gnar', 'gragas', 'graves', 'gwen', 'hecarim', 'heimerdinger', 'hwei', 'illaoi', 'irelia', 'ivern', 'janna', 'jarvaniv', 'jax', 'jayce', 'jhin', 'jinx', 'ksante', 'kaisa', 'kalista', 'karma', 'karthus', 'kassadin', 'katarina', 'kayle', 'kayn', 'kennen', 'khazix', 'kindred', 'kled', 'kogmaw', 'leblanc', 'leesin', 'leona', 'lillia', 'lissandra', 'lucian', 'lulu', 'lux', 'malphite', 'malzahar', 'maokai', 'masteryi', 'mel', 'milio', 'missfortune', 'mordekaiser', 'morgana', 'naafiri', 'nami', 'nasus', 'nautilus', 'neeko', 'nidalee', 'nilah', 'nocturne', 'nunu&willump', 'olaf', 'orianna', 'ornn', 'pa

In [ ]:
with open("champions.pkl", "rb") as champion_file:
    champions = pickle.load(champion_file)

In [2]:
import argparse
import json
import re
import time
from dataclasses import dataclass, asdict
from typing import Dict, Iterable, List, Optional, Tuple

from selenium import webdriver
from selenium.webdriver import ChromeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service


# -----------------------------
# Champion list (paste yours)
# -----------------------------
champions = [
    "Aatrox", "Ahri", "Akali", "Akshan", "Alistar", "Ambessa", "Amumu", "Anivia", "Annie", "Aphelios",
    "Ashe", "Aurelion Sol", "Aurora", "Azir", "Bard", "Belveth", "Blitzcrank", "Brand", "Braum", "Briar",
    "Caitlyn", "Camille", "Cassiopeia", "Chogath", "Corki", "Darius", "Diana", "Dr. Mundo", "Draven", "Ekko",
    "Elise", "Evelynn", "Ezreal", "Fiddlesticks", "Fiora", "Fizz", "Galio", "Gangplank", "Garen", "Gnar",
    "Gragas", "Graves", "Gwen", "Hecarim", "Heimerdinger", "Hwei", "Illaoi", "Irelia", "Ivern", "Janna",
    "Jarvan IV", "Jax", "Jayce", "Jhin", "Jinx", "K'Sante", "Kai'Sa", "Kalista", "Karma", "Karthus",
    "Kassadin", "Katarina", "Kayle", "Kayn", "Kennen", "Kha'Zix", "Kindred", "Kled", "Kog'Maw", "LeBlanc",
    "Lee Sin", "Leona", "Lillia", "Lissandra", "Lucian", "Lulu", "Lux", "Malphite", "Malzahar", "Maokai",
    "Master Yi", "Mel", "Milio", "Miss Fortune", "Mordekaiser", "Morgana", "Naafiri", "Nami", "Nasus",
    "Nautilus", "Neeko", "Nidalee", "Nilah", "Nocturne", "Nunu & Willump", "Olaf", "Orianna", "Ornn",
    "Pantheon", "Poppy", "Pyke", "Qiyana", "Quinn", "Rakan", "Rammus", "Rek'Sai", "Rell", "Renata Glasc",
    "Renekton", "Rengar", "Riven", "Rumble", "Ryze", "Samira", "Sejuani", "Senna", "Seraphine", "Sett",
    "Shaco", "Shen", "Shyvana", "Singed", "Sion", "Sivir", "Skarner", "Smolder", "Sona", "Soraka",
    "Swain", "Sylas", "Syndra", "Tahm Kench", "Taliyah", "Talon", "Taric", "Teemo", "Thresh", "Tristana",
    "Trundle", "Tryndamere", "Twisted Fate", "Twitch", "Udyr", "Urgot", "Varus", "Vayne", "Veigar",
    "Vel'Koz", "Vex", "Vi", "Viego", "Viktor", "Vladimir", "Volibear", "Warwick", "Wukong", "Xayah",
    "Xerath", "Xin Zhao", "Yasuo", "Yone", "Yorick", "Yunara", "Yuumi", "Zac", "Zed", "Zeri", "Ziggs",
    "Zilean", "Zoe", "Zyra"
]


In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://lolalytics.com/lol/ashe/build/"

response = requests.get(url)
response.raise_for_status()  # optional but recommended

soup = BeautifulSoup(response.text, "html.parser")


In [4]:
print(soup.get_text())

Ashe Build, Runes & Counters Guide for bottom AsheTier ListDiscordHomeTier ListLeaderboardCountersbottom0.6%0.0%0.6%92.8%5.9%EMERALD+Patch15.24RankedSolo/DuoGLOBALAverage Emerald+ Win Rate: 50%?Different tier brackets will have different average win rates so we now provide the average.This will vary because unlike other stat sites we include 100% of a brackets champions played and never include champions that don't belong in the bracket.Emerald+ Champions Analysed: 12,590,261?We are the only League of Legends stats site to analyse every champion from every ranked game.We include 100% of the champions played of every selected tier bracket in our analysis, we never include champions from outside the tier quoted tier.LoL Tier ListAshe BuildA89%Ashe Build, Runes & Counters for bottom AshePLoading...QLoading...WLoading...ELoading...RLoading...Ashe bottom has a 51.05% win rate in Emerald+ on Patch 15.24 coming in at rank 10 of 42 and graded A Tier on the LoL Tierlist. Ashe bottom is a strong

In [5]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en" q:base="/build/" q:container="paused" q:instance="lxgb7jabro" q:locale="" q:manifest-hash="83zmj1" q:render="ssr" q:route="lol/[champion]/build/" q:version="1.11.0">
 <!--qv q:id=0 q:key=J27P:PA_0-->
 <!--qv q:id=1 q:key=oI4p:HL_3-->
 <!--qv q:s q:sref=1 q:key=-->
 <head q:head="">
  <meta charset="utf-8" q:head=""/>
  <link href="/manifest.json" q:head="" rel="manifest"/>
  <!--qv q:id=2 q:key=Tsnx:HL_0-->
  <!--qv q:key=Vb_5-->
  <title q:head="">
   Ashe Build, Runes &amp; Counters Guide for bottom Ashe
  </title>
  <link href="https://lolalytics.com/lol/ashe/build/" q:head="" q:key="Vb_1" rel="canonical"/>
  <meta content="width=device-width, initial-scale=1, maximum-scale=5" name="viewport" q:head=""/>
  <!--qv q:key=Vb_3-->
  <link href="/favicon.ico" q:head="" rel="icon" type="image/svg+xml"/>
  <!--qv q:key=Vb_2-->
  <script data-cfasync="false" q:head="" q:key="Vb_0">
   window.nitroAds=window.nitroAds||{createAd:function(){return new Promise(e=

In [4]:
import os
import re
import time
import json
import pickle
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException, StaleElementReferenceException
import traceback
from pathlib import Path

def dump_debug(driver, slug, out_dir="lolalytics_out/debug"):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    ts = int(time.time())
    png = f"{out_dir}/{slug}_{ts}.png"
    html = f"{out_dir}/{slug}_{ts}.html"

    try:
        driver.save_screenshot(png)
    except Exception:
        pass

    try:
        with open(html, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
    except Exception:
        pass

    print(f"  [debug] saved {png} and {html}")


def scroll_into_view(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block:'center', inline:'nearest'});", element)

def click_weak_against(driver, timeout=20):
    """
    Clicks the 'Weak Against' filter and waits for the counters rows to render.
    """
    wait = WebDriverWait(driver, timeout)

    # Wait for the button to exist
    btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-type='weak_counter']")))
    scroll_into_view(driver, btn)
    time.sleep(0.25)

    # Click (JS click is more reliable on this site)
    try:
        wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div[data-type='weak_counter']")))
        btn.click()
    except (ElementClickInterceptedException, StaleElementReferenceException):
        driver.execute_script("arguments[0].click();", btn)

    # Wait until counters rows show up.
    # LoLalytics usually renders rows with champion links: /lol/<champ>/
    def rows_present(d):
        html = d.page_source
        return ("/lol/" in html) and ("Counter" in html or "Counters" in html)

    wait.until(rows_present)

@dataclass
class ScrapeConfig:
    base_url: str = "https://lolalytics.com/lol/{champ}/build/"
    lane: str = "bottom"          # e.g. top, jungle, middle, bottom, support
    tier: str = "emerald_plus"    # e.g. challenger, master, diamond_plus, emerald_plus, all
    region: str = "global"        # e.g. global
    patch: Optional[str] = None   # e.g. "15.24" or None for default site selection

    headless: bool = True
    timeout_sec: int = 25
    min_sleep: float = 0.8
    max_sleep: float = 1.8

    out_dir: str = "lolalytics_out"
    max_champs: Optional[int] = None   # set to e.g. 10 for testing


# Known URL “slug” exceptions (extend as you hit 404s)
SLUG_OVERRIDES = {
    "wukong": "wukong",
    "nunu & willump": "nunu",
    "dr. mundo": "drmundo",
    "kog'maw": "kogmaw",
    "kai'sa": "kaisa",
    "rek'sai": "reksai",
    "cho'gath": "chogath",
    "vel'koz": "velkoz",
    "kha'zix": "khazix",
    "bel'veth": "belveth",
    "lee sin": "leesin",
    "master yi": "masteryi",
    "miss fortune": "missfortune",
    "twisted fate": "twistedfate",
    "aurelion sol": "aurelionsol",
    "tahm kench": "tahmkench",
    "jarvan iv": "jarvaniv",
}


def load_champions(pkl_path: str = "champions.pkl") -> List[str]:
    with open(pkl_path, "rb") as f:
        champs = pickle.load(f)
    if not isinstance(champs, list):
        raise ValueError("champions.pkl must contain a Python list of champion names.")
    champs = [str(c).strip() for c in champs if str(c).strip()]
    return champs


def champ_to_slug(name: str) -> str:
    n = name.strip().lower()
    if n in SLUG_OVERRIDES:
        return SLUG_OVERRIDES[n]

    # generic slugify: keep letters/numbers, remove punctuation/spaces
    # lolalytics uses e.g. "kogmaw", "missfortune"
    n = n.replace("&", "and")
    n = re.sub(r"[^a-z0-9]+", "", n)
    return n

def build_url(champ_slug: str) -> str:
    return f"https://lolalytics.com/lol/{champ_slug}/build"




# -----------------------------
# Selenium setup
# -----------------------------

def make_driver(cfg: ScrapeConfig) -> webdriver.Chrome:
    opts = Options()
    if cfg.headless:
        opts.add_argument("--headless=new")

    # stability / CI-ish flags
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")

    # reduce botty vibes slightly
    opts.add_argument("--lang=en-US")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    # Selenium 4.6+ uses Selenium Manager (auto driver)
    driver = webdriver.Chrome(options=opts)
    driver.set_page_load_timeout(cfg.timeout_sec)
    return driver


def human_sleep(cfg: ScrapeConfig) -> None:
    time.sleep(random.uniform(cfg.min_sleep, cfg.max_sleep))


def try_accept_cookies(driver: webdriver.Chrome, timeout: int = 3) -> None:
    # This is best-effort; selectors can vary by region/time.
    possible_buttons = [
        (By.XPATH, "//button[contains(., 'Accept')]"),
        (By.XPATH, "//button[contains(., 'I agree')]"),
        (By.XPATH, "//button[contains(., 'OK')]"),
        (By.XPATH, "//button[contains(., 'Got it')]"),
    ]
    end = time.time() + timeout
    while time.time() < end:
        for by, sel in possible_buttons:
            try:
                btn = driver.find_element(by, sel)
                if btn.is_displayed():
                    btn.click()
                    return
            except Exception:
                pass
        time.sleep(0.2)


# -----------------------------
# Parsing helpers (works for <table> or div-grids)
# -----------------------------

def normalize_header(text: str) -> str:
    t = re.sub(r"\s+", " ", text.strip())
    t = t.replace("\u200b", "")
    return t


def extract_table_near_heading_html(soup: BeautifulSoup, heading_keywords: List[str]) -> Optional[str]:
    """
    Find a heading containing any keyword (case-insensitive) and return the HTML
    of the first table-like structure following it.
    """
    kws = [k.lower() for k in heading_keywords]

    # look for headings or section labels that contain the keywords
    candidates = soup.find_all(["h1", "h2", "h3", "div", "span"], string=True)
    target = None
    for c in candidates:
        txt = c.get_text(" ", strip=True).lower()
        if any(k in txt for k in kws):
            target = c
            break
    if not target:
        return None

    # try to find a <table> after the heading
    nxt = target
    for _ in range(60):
        nxt = nxt.find_next()
        if not nxt:
            break
        if nxt.name == "table":
            return str(nxt)

        # common "div table" patterns: role=table or role=rowgroup
        role = (nxt.get("role") or "").lower()
        if role == "table":
            return str(nxt)

        cls = " ".join(nxt.get("class", []))
        if "Table" in cls or "table" in cls.lower():
            # heuristic: a container with rows
            if nxt.find(attrs={"role": "row"}) or nxt.find("tr"):
                return str(nxt)

    return None


def parse_any_table(table_html: str) -> List[Dict[str, Any]]:
    """
    Parse either:
      - regular <table><thead><tbody><tr><td>...
      - div based tables using role=row/role=cell
    Returns list of dict rows.
    """
    s = BeautifulSoup(table_html, "html.parser")

    # Case A: real table
    table = s.find("table")
    if table:
        headers = []
        thead = table.find("thead")
        if thead:
            headers = [normalize_header(th.get_text(" ", strip=True)) for th in thead.find_all(["th", "td"])]
        # fallback: first row as header if looks like headers
        if not headers:
            first_tr = table.find("tr")
            if first_tr:
                headers = [normalize_header(x.get_text(" ", strip=True)) for x in first_tr.find_all(["th", "td"])]

        rows_out = []
        trs = table.find_all("tr")
        # if first row was used as header, skip it
        start_idx = 1 if trs and headers and len(trs[0].find_all(["th", "td"])) == len(headers) else 0

        for tr in trs[start_idx:]:
            cells = [normalize_header(td.get_text(" ", strip=True)) for td in tr.find_all(["td", "th"])]
            if not cells or all(c == "" for c in cells):
                continue
            row = {}
            for i, val in enumerate(cells):
                key = headers[i] if i < len(headers) else f"col_{i}"
                row[key] = val
            rows_out.append(row)
        return rows_out

    # Case B: role-based "div table"
    # Identify header row if present
    header_cells = []
    header_row = s.find(attrs={"role": "row"}, recursive=True)
    # some pages have multiple rowgroups; try to find a row with role=columnheader
    for r in s.find_all(attrs={"role": "row"}):
        colheads = r.find_all(attrs={"role": "columnheader"})
        if colheads:
            header_cells = [normalize_header(x.get_text(" ", strip=True)) for x in colheads]
            break

    # collect all data rows
    rows = []
    for r in s.find_all(attrs={"role": "row"}):
        # skip header-like rows
        if r.find(attrs={"role": "columnheader"}):
            continue
        cells = r.find_all(attrs={"role": "cell"})
        if not cells:
            # some layouts use divs without role=cell; fallback to direct children text chunks
            continue
        vals = [normalize_header(c.get_text(" ", strip=True)) for c in cells]
        if not vals or all(v == "" for v in vals):
            continue

        row = {}
        for i, v in enumerate(vals):
            key = header_cells[i] if i < len(header_cells) and header_cells[i] else f"col_{i}"
            row[key] = v
        rows.append(row)

    return rows


# -----------------------------
# Scrape one champ
# -----------------------------

def scrape_champ(driver, champ_name: str, cfg=None):
    slug = champ_to_slug(champ_name)
    url = f"https://lolalytics.com/lol/{slug}/build"

    driver.get(url)
    try_accept_cookies(driver)

    if "404" in (driver.title or "") or "Resource Not Found" in driver.page_source:
        raise RuntimeError(f"404 for {slug}")

    wait = WebDriverWait(driver, 25)
    weak_btn = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-type='weak_counter']"))
    )

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", weak_btn)
    time.sleep(0.25)
    driver.execute_script("arguments[0].click();", weak_btn)

    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "a[href^='/lol/'][href$='/']")) > 15)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    counters_html = extract_table_near_heading_html(soup, ["Counter", "Counters", "Matchups"])
    counters = parse_any_table(counters_html) if counters_html else []

    synergies_html = extract_table_near_heading_html(soup, ["Synergy", "Synergies", "Teammates"])
    synergies = parse_any_table(synergies_html) if synergies_html else []

    return {
        "champion": champ_name,
        "slug": slug,
        "url": url,
        "counters": counters,
        "synergies": synergies,
    }





# -----------------------------
# Run for all champs
# -----------------------------

def main():
    cfg = ScrapeConfig()

    os.makedirs(cfg.out_dir, exist_ok=True)

    champs = load_champions("champions.pkl")
    if cfg.max_champs:
        champs = champs[: cfg.max_champs]

    driver = make_driver(cfg)

    all_results: Dict[str, Any] = {}
    try:
        for i, champ in enumerate(champs, start=1):
            print(f"[{i}/{len(champs)}] {champ}")
            try:
                res = scrape_champ(driver, champ, cfg)
                all_results[res["slug"]] = res

                # write per-champ JSON (useful for incremental runs)
                out_path = os.path.join(cfg.out_dir, f"{res['slug']}.json")
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(res, f, ensure_ascii=False, indent=2)

            except Exception as e:
                print(f"  !! Failed {champ}: {type(e).__name__}: {repr(e)}")
                traceback.print_exc()
                dump_debug(driver, champ_to_slug(champ), out_dir=os.path.join(cfg.out_dir, "debug"))


            human_sleep(cfg)

    finally:
        driver.quit()

    # Write combined JSON
    combined_path = os.path.join(cfg.out_dir, "all_champions.json")
    with open(combined_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"Done. Output in: {cfg.out_dir}/")


if __name__ == "__main__":
    main()


[1/171] aatrox
[2/171] ahri
[3/171] akali
[4/171] akshan
[5/171] alistar


KeyboardInterrupt: 

In [1]:
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from typing import List, Dict, Any

# --- Configuration (remains the same) ---
CHAMPIONS_FILE = 'champions.pkl'
BASE_URL = 'https://lolalytics.com/lol/{}/build/'
OUTPUT_FILE = 'lolalytics_draft_data_adc.csv'
MAX_MATCHUPS_TO_SCRAPE = 10 
ROW_TIMEOUT = 5 # Max time to wait for a single row to appear

# XPATHS (remains the same)
WEAK_MATCHUPS_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[1]/div[2]/div/div[3]"
COOKIE_ACCEPT_BUTTON_XPATH = "//button[contains(text(), 'Accept')]" 
DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[5]/div[2]/div/div" 
FIRST_ADC_CHAMP_NAME_XPATH = f"{DATA_BASE_XPATH}[1]/a"

def setup_driver():
    """Initializes the Chrome WebDriver."""
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') # Uncomment for invisible operation
    options.add_argument('--log-level=3') # Suppress console logs
    driver = webdriver.Chrome(service=Service(), options=options)
    return driver

def load_champions(filename: str) -> List[str]:
    """Loads the list of champions from a pickle file."""
    try:
        with open(filename, 'rb') as f:
            champs = pickle.load(f)
            return [str(c) for c in champs]
    except FileNotFoundError:
        print(f"Error: Champion file '{filename}' not found. Please create it first.")
        return []
    except Exception as e:
        print(f"An error occurred loading champions: {e}")
        return []
    
def format_champion_name(name: str) -> str:
    """Formats champion names for the Lolalytics URL."""
    return name.lower().replace(" ", "").replace("'", "")

In [ ]:
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import pandas as pd
from typing import List, Dict, Any

# --- Configuration (Optimized settings remain the same) ---
CHAMPIONS_FILE = 'champions.pkl'
BASE_URL = 'https://lolalytics.com/lol/{}/build/'
OUTPUT_FILE = 'lolalytics_draft_data_combined.csv'
MAX_MATCHUPS_TO_SCRAPE = 25 
ROW_TIMEOUT = 3 

# --- NEW: SUPPORT CHAMPION LIST FOR RELIABLE LOGIC ---
# This list covers most dedicated and commonly played Support champions.
SUPPORT_CHAMPS = {
    'Alistar', 'Bard', 'Blitzcrank', 'Braum', 'Janna', 'Karma', 'Leona', 'Lulu', 
    'Lux', 'Maokai', 'Milio', 'Morgana', 'Nami', 'Nautilus', 'Pantheon', 'Pyke', 
    'Rakan', 'Rell', 'Renata Glasc', 'Senna', 'Seraphine', 'Sett', 'Shaco', 
    'Sona', 'Soraka', 'Swain', 'Tahm Kench', 'Taric', 'Thresh', 'Yuumi', 'Zac',
    'Zilean', 'Zyra', 'Heimerdinger'
}
SUPPORT_CHAMPS = [support_champ.lower() for support_champ in SUPPORT_CHAMPS]

# --- XPATHS ---
COOKIE_ACCEPT_BUTTON_XPATH = "//button[contains(text(), 'Accept')]" 
WEAK_MATCHUPS_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[1]/div[2]/div/div[3]"
GOOD_SYNERGIES_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[2]/div[2]/div[2]"

# DISTINCT BASE XPATHS
SYNERGY_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[4]/div[2]/div/div"  # div[4] for non-support
MATCHUP_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[5]/div[2]/div/div"  # div[5] for weak matchups AND support synergies

# Elements used for waiting/checking
BUILD_SUMMARY_XPATH = "/html/body/main/div[6]/div[1]/div[2]" 
THIRD_MATCHUP_CHAMP_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[3]/a/span/img"


# --- Helper Functions (setup_driver, load_champions, format_champion_name remain the same) ---
def setup_driver():
    """Initializes the Chrome WebDriver in VISIBLE MODE (for reliability)."""
    options = webdriver.ChromeOptions()
    options.add_argument('--log-level=3') 
    driver = webdriver.Chrome(service=Service(), options=options)
    return driver

def load_champions(filename: str) -> List[str]:
    """Loads the list of champions from a pickle file."""
    try:
        with open(filename, 'rb') as f:
            champs = pickle.load(f)
            return [str(c) for c in champs]
    except FileNotFoundError:
        print(f"Error: Champion file '{filename}' not found. Please create it first.")
        return []
    except Exception as e:
        print(f"An error occurred loading champions: {e}")
        return []

def format_champion_name(name: str) -> str:
    """Formats champion names for the Lolalytics URL."""
    return name.lower().replace(" ", "").replace("'", "")

def scrape_data(driver: webdriver.Chrome, champion: str) -> List[Dict[str, Any]]:
    """Navigates, clicks, scrolls, and scrapes both Weak Matchups and Good Synergies."""
    formatted_champ = format_champion_name(champion)
    url = BASE_URL.format(formatted_champ)
    print(f"\n--- Scraping Matchups and Synergies for: {champion} (Max {MAX_MATCHUPS_TO_SCRAPE} each) ---")
    
    driver.get(url)
    wait = WebDriverWait(driver, 20)
    champion_data = []

    # 1. INITIAL PAGE LOAD & COOKIE HANDLER (Same as before)
    print(" -> Waiting for core page content to load...")
    try:
        wait.until(EC.presence_of_element_located((By.XPATH, BUILD_SUMMARY_XPATH)))
        try:
            cookie_button = wait.until(EC.element_to_be_clickable((By.XPATH, COOKIE_ACCEPT_BUTTON_XPATH)))
            driver.execute_script("arguments[0].click();", cookie_button)
        except Exception:
            pass
    except Exception as e:
        print(f" -> ERROR: Initial page content or cookie banner failed to load quickly. Error: {e}")
        return []
    
    # 2. Initial Scroll Down (Same as before)
    print(" -> Scrolling down to load dynamic content area...")
    driver.execute_script("window.scrollTo(0, 1500);")
    
    # --- BLOCK 1: WEAK MATCHUPS (Uses div[5] - Retry logic ensures extraction) ---
    print("\n[WEAK MATCHUPS]")
    max_attempts = 2 
    attempt = 0
    matchup_success = False

    while attempt < max_attempts and not matchup_success:
        attempt += 1
        print(f" -> Attempt {attempt}: Clicking Weak Matchups tab via JS...")
        try:
            weak_matchups_tab = wait.until(EC.presence_of_element_located((By.XPATH, WEAK_MATCHUPS_TAB_XPATH)))
            driver.execute_script("arguments[0].click();", weak_matchups_tab)
            
            print(" -> Pausing 1s after click for UI script execution...")
            time.sleep(1)
            
            print(" -> Waiting up to 8s for the top of the matchup list to render fully (div[5])...")
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.XPATH, THIRD_MATCHUP_CHAMP_XPATH))
            )
            matchup_success = True
        except Exception:
            if attempt < max_attempts:
                print(" -> Matchup rendering failed. Retrying...")
                time.sleep(1) 
            else:
                print(" -> Matchup rendering failed after all attempts. Skipping.")
                
    
    if matchup_success:
        extracted_count = 0
        for i in range(1, MAX_MATCHUPS_TO_SCRAPE + 1):
            CHAMP_LINK_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[{i}]/a"
            CHAMP_IMG_XPATH = f"{CHAMP_LINK_XPATH}/span/img"
            WIN_RATE_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[{i}]/div[1]/span"
            
            try:
                WebDriverWait(driver, ROW_TIMEOUT).until(
                    EC.presence_of_element_located((By.XPATH, CHAMP_IMG_XPATH))
                )

                champ_img_element = driver.find_element(By.XPATH, CHAMP_IMG_XPATH)
                matchup_champ = champ_img_element.get_attribute("alt")
                value_element = driver.find_element(By.XPATH, WIN_RATE_XPATH)
                win_rate_delta = value_element.text
                
                if matchup_champ and win_rate_delta:
                    champion_data.append({
                        'champion': champion, 
                        'category': 'WEAK_MATCHUP',
                        'vs_champ': matchup_champ, 
                        'value': win_rate_delta 
                    })
                    extracted_count += 1
                
                if i >= 2 and i % 5 == 0 and i < MAX_MATCHUPS_TO_SCRAPE: 
                    next_element_xpath = f"{MATCHUP_DATA_BASE_XPATH}[{i+1}]"
                    try:
                        WebDriverWait(driver, 1).until(
                            EC.presence_of_element_located((By.XPATH, next_element_xpath))
                        )
                        next_element = driver.find_element(By.XPATH, next_element_xpath)
                        driver.execute_script("arguments[0].scrollIntoView(false);", next_element)
                    except Exception:
                        pass

            except Exception:
                break 
        
        print(f" -> Successfully extracted {extracted_count} weak matchup rows.")

    # --- BLOCK 2: GOOD SYNERGIES (NEW SUPPORT LIST LOGIC) ---
    print("\n[GOOD SYNERGIES]")
    try:
        print(" -> Clicking Good Synergies tab via JS...")
        good_synergies_tab = wait.until(EC.presence_of_element_located((By.XPATH, GOOD_SYNERGIES_TAB_XPATH)))
        driver.execute_script("arguments[0].click();", good_synergies_tab)
        
        # Targeted wait after click
        time.sleep(1)

        # --- DETERMINE DATA BLOCK XPATH BASED ON CHAMPION ROLE ---
        if champion in SUPPORT_CHAMPS:
            synergy_base_path = MATCHUP_DATA_BASE_XPATH # Use div[5]
            print(f" -> {champion} is a SUPPORT: Using MATCHUP_DATA_BASE_XPATH (div[5]).")
        else:
            synergy_base_path = SYNERGY_DATA_BASE_XPATH # Use div[4]
            print(f" -> {champion} is NON-SUPPORT: Using SYNERGY_DATA_BASE_XPATH (div[4]).")

        # Wait for the content to swap again using the determined base path
        FIRST_SYNERGY_CHAMP_XPATH = f"{synergy_base_path}[1]/a"
        try:
            WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, FIRST_SYNERGY_CHAMP_XPATH)))
        except TimeoutException:
            print(" -> WARNING: Synergy data failed initial load check.")


        extracted_count = 0
        for i in range(1, MAX_MATCHUPS_TO_SCRAPE + 1):
            # *** USE THE DYNAMICALLY DETERMINED synergy_base_path ***
            CHAMP_LINK_XPATH = f"{synergy_base_path}[{i}]/a"
            CHAMP_IMG_XPATH = f"{CHAMP_LINK_XPATH}/span/img"
            WIN_RATE_XPATH = f"{synergy_base_path}[{i}]/div[1]/span"
            
            try:
                WebDriverWait(driver, ROW_TIMEOUT).until(
                    EC.presence_of_element_located((By.XPATH, CHAMP_IMG_XPATH))
                )

                champ_img_element = driver.find_element(By.XPATH, CHAMP_IMG_XPATH)
                synergy_champ = champ_img_element.get_attribute("alt")
                value_element = driver.find_element(By.XPATH, WIN_RATE_XPATH)
                synergy_delta = value_element.text
                
                if synergy_champ and synergy_delta:
                    champion_data.append({
                        'champion': champion, 
                        'category': 'GOOD_SYNERGY', 
                        'with_champ': synergy_champ, 
                        'value': synergy_delta 
                    })
                    extracted_count += 1

                # SCROLL OPTIMIZATION
                if i >= 2 and i % 5 == 0 and i < MAX_MATCHUPS_TO_SCRAPE:
                    next_element_xpath = f"{synergy_base_path}[{i+1}]"
                    try:
                        WebDriverWait(driver, 1).until(
                            EC.presence_of_element_located((By.XPATH, next_element_xpath))
                        )
                        next_element = driver.find_element(By.XPATH, next_element_xpath)
                        driver.execute_script("arguments[0].scrollIntoView(false);", next_element)
                    except Exception:
                        pass
                
            except Exception:
                print(f" -> Stopped extraction after {i-1} synergy rows.")
                break 
        
        print(f" -> Successfully extracted {extracted_count} synergy rows.")
        
    except Exception as e:
        print(f" -> ERROR: Good Synergy scrape failed: {e}")
        
    return champion_data

# The run_scraper function remains the same.

def run_scraper():
    """Main execution function. Sets up and quits the driver for *each* champion."""
    champions = load_champions(CHAMPIONS_FILE)
    if not champions:
        return

    all_data = []

    try:
        champions_to_scrape = champions 

        for champ in champions_to_scrape:
            # 1. SETUP DRIVER
            driver = setup_driver()
            
            # 2. SCRAPE
            data = scrape_data(driver, champ)
            all_data.extend(data)

            try:
                df = pd.DataFrame(all_data)
                df.to_excel("lolalytics_data.xlsx")
            except Exception as e:
                print(f"Error! {e}")
            
            # 3. QUIT DRIVER immediately to free resources and flush cache
            driver.quit()
            
            # Small, non-critical wait between driver starts
            time.sleep(1) 
            
    except Exception as e:
        print(f"An unexpected error occurred in the main loop: {e}")

    # Convert to DataFrame and save
    if all_data:
        df = pd.DataFrame(all_data)
        df.to_csv(OUTPUT_FILE, index=False)
        print("\n--- Scraper Finished ---")
        print(f"Data successfully scraped for {len(df['champion'].unique())} champions and saved to {OUTPUT_FILE}")
        print("\nExample Data:")
        print(df.head())
    else:
        print("WARNING: No data was successfully scraped.")

if __name__ == '__main__':
    run_scraper()
    print("The scraper is now using role-based logic for synergy XPATHS. If the champion being scraped is in the predefined list of SUPPORT_CHAMPS, it uses the div[5] block for synergies; otherwise, it uses the standard div[4] block.")


--- Scraping Matchups and Synergies for: aatrox (Max 25 each) ---
 -> Waiting for core page content to load...
 -> Scrolling down to load dynamic content area...

[WEAK MATCHUPS]
 -> Attempt 1: Clicking Weak Matchups tab via JS...
 -> Pausing 1s after click for UI script execution...
 -> Waiting up to 8s for the top of the matchup list to render fully (div[5])...
 -> Successfully extracted 25 weak matchup rows.

[GOOD SYNERGIES]
 -> Clicking Good Synergies tab via JS...
 -> aatrox is NON-SUPPORT: Using SYNERGY_DATA_BASE_XPATH (div[4]).
 -> Successfully extracted 25 synergy rows.

--- Scraping Matchups and Synergies for: ahri (Max 25 each) ---
 -> Waiting for core page content to load...
 -> Scrolling down to load dynamic content area...

[WEAK MATCHUPS]
 -> Attempt 1: Clicking Weak Matchups tab via JS...
 -> Pausing 1s after click for UI script execution...
 -> Waiting up to 8s for the top of the matchup list to render fully (div[5])...
 -> Successfully extracted 25 weak matchup rows.

In [6]:
df = pd.read_csv("lolalytics_draft_data_combined.csv")
df.to_excel("lolalytics_data.xlsx")

In [58]:
def calculate_optimal_pick(df: pd.DataFrame, allies: list, enemies: list):
    """
    Calculates the combined Win Rate Delta for every champion against a specific draft.
    The highest score indicates the most optimal pick.
    """
    
    try:
        df['value_numeric'] = pd.to_numeric(df['value'], errors='coerce')
        # Convert win rate (e.g., 51.86) to delta (e.g., +1.86)
        df['delta'] = df['value_numeric'] - 50.0
        df.dropna(subset=['delta'], inplace=True)
    except Exception as e:
        print(f"Error converting 'value' column to numeric: {e}. Check your data format.")
        return None

    # Rename columns for consistent merging/grouping
    df['matchup_champ'] = df['vs_champ'].fillna(df['with_champ'])
    
    # 2. Filter data for the current draft
    
    # Champions where the selected pick is an enemy (WEAK_MATCHUP)
    enemy_matchups = df[
        (df['category'] == 'WEAK_MATCHUP') & (df['champion'].isin(enemies))
    ].copy()
    # print(f"enemy_matchups: {enemy_matchups}")
    
    # Champions where the selected pick is an ally (GOOD_SYNERGY)
    ally_synergies = df[
        (df['category'] == 'GOOD_SYNERGY') & (df['champion'].isin(allies))
    ].copy()

    matchup_dict = {}
    for index, enemy_matchup in enemy_matchups.iterrows():
        if enemy_matchup["vs_champ"] in matchup_dict.keys():
            matchup_dict[enemy_matchup["vs_champ"]].append(100-enemy_matchup["value"])
        else:
            matchup_dict[enemy_matchup["vs_champ"]] = [100-enemy_matchup["value"]]

    for index, synergy in ally_synergies.iterrows():
        if synergy["with_champ"] in matchup_dict.keys():
            matchup_dict[synergy["with_champ"]].append(synergy["value"])
        else:
            matchup_dict[synergy["with_champ"]] = [synergy["value"]]
    for key, values in matchup_dict.items():
        matchup_dict[key] = round(sum(values) / len(values), 2)

    return matchup_dict

adcs = [
    "Aphelios",
    "Ashe",
    "Caitlyn",
    "Corki",
    "Draven",
    "Ezreal",
    "Graves",
    "Jhin",
    "Jinx",
    "Kai'Sa",
    "Kalista",
    "Kindred",
    "Kog'Maw",
    "Lucian",
    "Miss Fortune",
    "Nilah",
    "Quinn",
    "Samira",
    "Senna",
    "Sivir",
    "Smolder",
    "Tristana",
    "Twitch",
    "Varus",
    "Vayne",
    "Xayah",
    "Zeri",
    "Swain",
    "Seraphine",
]
df = pd.read_excel("lolalytics_data.xlsx")
df = pd.read_csv("lolalytics_draft_data_combined.csv")
allied_champs = []
enemy_champs = ["Ekko"]
enemy_champs = [enemy_champ.lower() for enemy_champ in enemy_champs]
allied_champs = [allied_champs.lower() for allied_champs in allied_champs]
optimal_champs = calculate_optimal_pick(df, allied_champs, enemy_champs)
optimal_champs = {key: value for key, value in optimal_champs.items() if key in adcs}
print(sorted(optimal_champs.items(), key=lambda item: item[1], reverse=True))


[('Miss Fortune', 51.85), ('Swain', 51.65), ('Samira', 51.18), ('Nilah', 51.16), ('Draven', 50.8), ('Tristana', 50.63), ('Vayne', 50.37), ('Jinx', 50.02), ('Ashe', 49.56), ('Jhin', 49.32), ('Aphelios', 48.95), ('Twitch', 48.95), ('Caitlyn', 48.81), ('Smolder', 48.63), ("Kai'Sa", 48.52), ('Sivir', 48.5), ('Lucian', 48.44), ('Xayah', 48.22), ('Varus', 47.48), ('Kalista', 47.41), ('Ezreal', 47.14), ('Zeri', 45.74)]
